In [3]:
import geopandas as gpd
import pandas as pd

In [41]:
glake = gpd.read_file(r'model_data\Final data\lakes_imputed_22.gpkg')
# lake_clean = gpd.read_file(r'lakes_clean.gpkg')

In [42]:
print("Glake columns:", glake.columns)  
# print("Lake_clean columns:", lake_clean.columns)

Glake columns: Index(['Type', 'Elevation', 'GLAKE_ID', 'area_2020', 'perimeter_2020',
       'area_1990', 'perimeter_1990', 'geometry_1990', 'area_2000',
       'perimeter_2000', 'geometry_2000', 'area_2010', 'perimeter_2010',
       'geometry_2010', 'area_2015', 'perimeter_2015', 'geometry_2015',
       'area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2',
       'area_2020_km2', 'expansion_rate_km2_yr', 'expansion_rate_pct_yr',
       'glof_happened', 'glof_count', 'lake_type', 'distance_to_glacier_m',
       'nearest_rgiid', 'is_connected', 'wse_m_dam', 'eq_susceptibility',
       'ls_susceptibility', 'volume_m3', 'max_depth_m', 'surface_elevation_m',
       'area_m2', 'perimeter_m', 'RGIId', 'G11_mean_slope_deg',
       'area_exp_1990_2000', 'area_exp_pct_1990_2000',
       'area_cagr_pct_1990_2000', 'area_exp_2000_2010',
       'area_exp_pct_2000_2010', 'area_cagr_pct_2000_2010',
       'area_exp_2010_2015', 'area_exp_pct_2010_2015',
       'area_cagr_pct_2010_2015',

In [5]:
# Peak area across all time steps — captures "was dangerous in the past"
area_cols = ['area_1990', 'area_2000', 'area_2010', 
             'area_2015', 'area_2020']
glake['peak_area_km2'] = glake[area_cols].max(axis=1)
glake['area_change_total'] = glake['area_2020'] - glake['area_1990']

# Also: year of peak area — did the lake shrink after GLOF?
glake['peak_area_year'] = glake[area_cols].idxmax(axis=1).str.extract(
    r'(\d{4})').astype(float)

In [27]:
# Check what your missing values look like
area_cols = ['area_1990_km2', 'area_2000_km2', 'area_2010_km2',
             'area_2015_km2', 'area_2020_km2']

print(glake[area_cols].isna().sum())
print("\nExample rows with missing area values:")
print(glake[area_cols].head(20).to_string())

area_1990_km2    1428
area_2000_km2     965
area_2010_km2     601
area_2015_km2     333
area_2020_km2       9
dtype: int64

Example rows with missing area values:
    area_1990_km2  area_2000_km2  area_2010_km2  area_2015_km2  area_2020_km2
0        0.106050       0.113754       0.111538       0.071794       0.068532
1        0.038557       0.072417       0.047025       0.060218       0.058420
2        0.129659       0.139595       0.171644       0.208555       0.223837
3             NaN       0.022145       0.030620       0.067422       0.074128
4        0.038415       0.043786       0.087872       0.118966       0.169969
5        0.163041       0.168617       0.170234       0.147493       0.150567
6        0.013736       0.011094       0.012522       0.018608       0.010801
7        0.368131       0.309572       0.408826       0.434772       0.416247
8             NaN            NaN            NaN       0.043583       0.017837
9             NaN            NaN       0.023705       0.0

In [25]:
# Conservative temporal filling for glacial-lake area time series.
# No extra flag columns required.

def fill_temporal_area(
    row,
    cols,
    fill_endpoints=False,
    interpolate_inside=True,
):
    """
    Rules:
    - All-null series: keep as NaN (unmapped/unknown).
    - Interior gaps: linear interpolation only (optional).
    - Endpoint zeros are optional and OFF by default.
    - Never allow negative area.

    Parameters
    ----------
    row : pd.Series
        One lake record.
    cols : list[str]
        Ordered temporal area columns.
    fill_endpoints : bool
        If True, fill years before first and after last observed value with 0.0.
        Keep False for conservative behavior when absence is uncertain.
    interpolate_inside : bool
        If True, fill only interior gaps by linear interpolation.
    """
    s = pd.Series([row[c] for c in cols], index=cols, dtype=float)

    if s.isna().all():
        return s

    first_valid = s.first_valid_index()
    last_valid = s.last_valid_index()

    first_idx = list(cols).index(first_valid)
    last_idx = list(cols).index(last_valid)

    if fill_endpoints:
        s.iloc[:first_idx] = 0.0
        s.iloc[last_idx + 1:] = 0.0

    if interpolate_inside:
        s = s.interpolate(method='linear', limit_area='inside')

    s = s.clip(lower=0.0)
    return s

# Usage without any absence-flag columns:
# Conservative default: do not force endpoint zeros
# (set fill_endpoints=True only if you want that assumption).
glake[area_cols] = glake.apply(
    lambda r: fill_temporal_area(
        r,
        area_cols,
        fill_endpoints=False,
        interpolate_inside=True,
    ),
    axis=1,
    result_type='expand'
)

In [26]:
glake.columns

Index(['Type', 'Elevation', 'GLAKE_ID', 'area_2020', 'perimeter_2020',
       'area_1990', 'perimeter_1990', 'geometry_1990', 'area_2000',
       'perimeter_2000', 'geometry_2000', 'area_2010', 'perimeter_2010',
       'geometry_2010', 'area_2015', 'perimeter_2015', 'geometry_2015',
       'area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2',
       'area_2020_km2', 'expansion_rate_km2_yr', 'expansion_rate_pct_yr',
       'glof_happened', 'glof_count', 'lake_type', 'distance_to_glacier_m',
       'nearest_rgiid', 'is_connected', 'dam_crest_m_dam', 'dam_height_m_dam',
       'dam_slope_deg_dam', 'dam_width_m_dam', 'freeboard_m_dam', 'wse_m_dam',
       'eq_susceptibility', 'ls_susceptibility', 'volume_m3', 'max_depth_m',
       'surface_elevation_m', 'area_m2', 'perimeter_m', 'RGIId',
       'G11_mean_slope_deg', 'geometry'],
      dtype='object')

In [29]:
import numpy as np

# Area expansion features from time-ordered area columns.
# Works for names like area_1990, area_1990_km2, etc.

years = [int(pd.Series([c]).str.extract(r'(\d{4})').iloc[0, 0]) for c in area_cols]

# Pairwise interval expansion features (absolute, percent, annualized)
for i in range(1, len(area_cols)):
    c_prev, c_curr = area_cols[i - 1], area_cols[i]
    y_prev, y_curr = years[i - 1], years[i]
    dt = y_curr - y_prev

    # Absolute expansion over interval
    glake[f'area_exp_{y_prev}_{y_curr}'] = glake[c_curr] - glake[c_prev]

    # Percent expansion over interval (safe divide)
    denom = glake[c_prev].replace(0, np.nan)
    glake[f'area_exp_pct_{y_prev}_{y_curr}'] = (
        (glake[c_curr] - glake[c_prev]) / denom
    ) * 100.0

    # Annualized growth rate in percent (CAGR-style, safe for non-positive baseline)
    valid = (glake[c_prev] > 0) & (glake[c_curr] >= 0)
    glake[f'area_cagr_pct_{y_prev}_{y_curr}'] = np.nan
    glake.loc[valid, f'area_cagr_pct_{y_prev}_{y_curr}'] = (
        (glake.loc[valid, c_curr] / glake.loc[valid, c_prev]) ** (1.0 / dt) - 1.0
    ) * 100.0

# Overall expansion from first to last year in area_cols
first_col, last_col = area_cols[0], area_cols[-1]
first_year, last_year = years[0], years[-1]

# Net absolute and percent expansion
glake['area_exp_total'] = glake[last_col] - glake[first_col]
base = glake[first_col].replace(0, np.nan)
glake['area_exp_total_pct'] = ((glake[last_col] - glake[first_col]) / base) * 100.0

# Annualized expansion across full time window
dt_total = last_year - first_year
valid_total = (glake[first_col] > 0) & (glake[last_col] >= 0)
glake['area_cagr_total_pct'] = np.nan
glake.loc[valid_total, 'area_cagr_total_pct'] = (
    (glake.loc[valid_total, last_col] / glake.loc[valid_total, first_col]) ** (1.0 / dt_total) - 1.0
) * 100.0

# Trend summary: how often area increased/decreased between snapshots
delta_matrix = glake[area_cols].diff(axis=1).iloc[:, 1:]
glake['n_expanding_intervals'] = (delta_matrix > 0).sum(axis=1)
glake['n_shrinking_intervals'] = (delta_matrix < 0).sum(axis=1)

# Clean numeric artifacts from division
new_cols = [c for c in glake.columns if c.startswith('area_exp_') or c.startswith('area_cagr_')]
glake[new_cols] = glake[new_cols].replace([np.inf, -np.inf], np.nan)

show_cols = [
    'area_exp_total', 'area_exp_total_pct', 'area_cagr_total_pct',
    'n_expanding_intervals', 'n_shrinking_intervals'
]
print(glake[show_cols].head())

   area_exp_total  area_exp_total_pct  area_cagr_total_pct  \
0       -0.037518          -35.378057            -1.444848   
1        0.019863           51.516044             1.394708   
2        0.094178           72.635145             1.836698   
3             NaN                 NaN                  NaN   
4        0.131554          342.454202             5.082149   

   n_expanding_intervals  n_shrinking_intervals  
0                      1                      3  
1                      2                      2  
2                      4                      0  
3                      3                      0  
4                      4                      0  


In [43]:
glake.columns

Index(['Type', 'Elevation', 'GLAKE_ID', 'area_2020', 'perimeter_2020',
       'area_1990', 'perimeter_1990', 'geometry_1990', 'area_2000',
       'perimeter_2000', 'geometry_2000', 'area_2010', 'perimeter_2010',
       'geometry_2010', 'area_2015', 'perimeter_2015', 'geometry_2015',
       'area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2',
       'area_2020_km2', 'expansion_rate_km2_yr', 'expansion_rate_pct_yr',
       'glof_happened', 'glof_count', 'lake_type', 'distance_to_glacier_m',
       'nearest_rgiid', 'is_connected', 'wse_m_dam', 'eq_susceptibility',
       'ls_susceptibility', 'volume_m3', 'max_depth_m', 'surface_elevation_m',
       'area_m2', 'perimeter_m', 'RGIId', 'G11_mean_slope_deg',
       'area_exp_1990_2000', 'area_exp_pct_1990_2000',
       'area_cagr_pct_1990_2000', 'area_exp_2000_2010',
       'area_exp_pct_2000_2010', 'area_cagr_pct_2000_2010',
       'area_exp_2010_2015', 'area_exp_pct_2010_2015',
       'area_cagr_pct_2010_2015', 'area_exp_2015

In [31]:
glake['n_expanding_intervals']

0       1
1       2
2       4
3       3
4       4
       ..
6918    3
6919    3
6920    3
6921    3
6922    3
Name: n_expanding_intervals, Length: 6923, dtype: int64

In [38]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, cross_validate

# ---------------------------------------------------------
# 1) Set target and basic column exclusions
# ---------------------------------------------------------
# If your target differs, change this to your label column.
TARGET_COL = 'glof_happened'

candidate_targets = [
    'glof_occurred', 'glof', 'label', 'target', 'y', 'class', 'susceptibility_class'
]

if TARGET_COL is None:
    for c in candidate_targets:
        if c in glake.columns:
            TARGET_COL = c
            break

if TARGET_COL is None or TARGET_COL not in glake.columns:
    print('Target column not found in dataset.')
    print('Set TARGET_COL to one of your actual columns, for example "glof_happened".')
    print('Candidate-like columns in data:')
    maybe = [c for c in glake.columns if any(k in c.lower() for k in ['glof', 'target', 'label', 'class', 'suscept'])]
    print(maybe if maybe else 'None found by keyword scan.')
else:
    drop_cols = [TARGET_COL]
    if 'geometry' in glake.columns:
        drop_cols.append('geometry')

    df = glake.drop(columns=drop_cols, errors='ignore').copy()
    y = glake[TARGET_COL].copy()

    # Keep only rows with non-null target
    valid_y = y.notna()
    df = df.loc[valid_y].copy()
    y = y.loc[valid_y]

    # Ensure binary integer target if it is boolean-like/object-like
    if y.dtype == bool:
        y = y.astype(int)

    # Optional: add missingness indicators for ratio-like engineered columns
    ratio_like = [c for c in df.columns if ('pct' in c.lower() or 'cagr' in c.lower())]
    for c in ratio_like:
        df[f'{c}_was_nan'] = df[c].isna().astype('int8')

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

    # Pre-clean numeric matrix for inf/overflow from ratio features
    df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)

    # Soft clipping to reduce extreme outliers that can destabilize some models
    if len(num_cols) > 0:
        q_lo = df[num_cols].quantile(0.001)
        q_hi = df[num_cols].quantile(0.999)
        df[num_cols] = df[num_cols].clip(lower=q_lo, upper=q_hi, axis=1)

    print(f'Using target: {TARGET_COL}')
    print(f'Rows: {len(df)}, Numeric cols: {len(num_cols)}, Categorical cols: {len(cat_cols)}')

    # ---------------------------------------------------------
    # 2) Define imputers and models to compare
    # ---------------------------------------------------------
    imputers = {
        'simple_median': SimpleImputer(strategy='median'),
        'knn': KNNImputer(n_neighbors=5),
        'iterative': IterativeImputer(random_state=42, max_iter=20)
    }

    models = {
        'logreg': LogisticRegression(max_iter=2000, class_weight='balanced'),
        'rf': RandomForestClassifier(n_estimators=500, random_state=42, class_weight='balanced'),
        'xgb': XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric='logloss',
            random_state=42
        )
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scoring = {'ap': 'average_precision', 'roc_auc': 'roc_auc'}

    results = []

    # ---------------------------------------------------------
    # 3) Benchmark all imputer-model combinations
    # ---------------------------------------------------------
    for imp_name, imp in imputers.items():
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', Pipeline([('imputer', imp)]), num_cols),
                ('cat', Pipeline([
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore'))
                ]), cat_cols)
            ],
            remainder='drop'
        )

        for model_name, model in models.items():
            pipe = Pipeline([
                ('prep', preprocessor),
                ('model', model)
            ])

            try:
                cv_out = cross_validate(
                    pipe,
                    df,
                    y,
                    cv=cv,
                    scoring=scoring,
                    n_jobs=-1,
                    error_score='raise'
                )

                results.append({
                    'imputer': imp_name,
                    'model': model_name,
                    'ap_mean': np.mean(cv_out['test_ap']),
                    'ap_std': np.std(cv_out['test_ap']),
                    'roc_auc_mean': np.mean(cv_out['test_roc_auc']),
                    'roc_auc_std': np.std(cv_out['test_roc_auc']),
                    'status': 'ok'
                })
            except Exception as e:
                results.append({
                    'imputer': imp_name,
                    'model': model_name,
                    'ap_mean': np.nan,
                    'ap_std': np.nan,
                    'roc_auc_mean': np.nan,
                    'roc_auc_std': np.nan,
                    'status': f'failed: {str(e)[:120]}'
                })

    results_df = pd.DataFrame(results).sort_values(['ap_mean', 'roc_auc_mean'], ascending=False, na_position='last')
    print('\nBenchmark results:')
    print(results_df.to_string(index=False))

    # Optional: keep for later use
    benchmark_results = results_df

Using target: glof_happened
Rows: 6923, Numeric cols: 57, Categorical cols: 15

Benchmark results:
      imputer  model  ap_mean   ap_std  roc_auc_mean  roc_auc_std                                                                                                                           status
simple_median    xgb 0.977940 0.044120      0.980265     0.039470                                                                                                                               ok
          knn    xgb 0.977933 0.044134      0.979328     0.041344                                                                                                                               ok
simple_median     rf 0.918961 0.121800      0.984849     0.030075                                                                                                                               ok
          knn     rf 0.916287 0.132423      0.984615     0.030543                                                        

In [40]:
results_df

,imputer,model,ap_mean,ap_std,roc_auc_mean,roc_auc_std,status
2,simple_median,xgb,0.977940,0.044120,0.980265,0.039470,ok
5,knn,xgb,0.977933,0.044134,0.979328,0.041344,ok
1,simple_median,rf,0.918961,0.121800,0.984849,0.030075,ok
4,knn,rf,0.916287,0.132423,0.984615,0.030543,ok
6,iterative,logreg,0.105193,0.085030,0.597264,0.158274,ok
0,simple_median,logreg,0.094195,0.090938,0.695522,0.071359,ok
3,knn,logreg,0.046842,0.042905,0.667729,0.109435,ok
7,iterative,rf,NaN,NaN,NaN,NaN,failed: Input X contains infinity or a value t...
8,iterative,xgb,NaN,NaN,NaN,NaN,failed: [15:23:01] D:\bld\xgboost-split_177212...


In [41]:
import re
import numpy as np
from sklearn.impute import SimpleImputer

# 1) Median-impute all numeric columns in glake that contain missing values
num_cols = glake.select_dtypes(include=[np.number]).columns.tolist()
num_missing_cols = [c for c in num_cols if glake[c].isna().any()]

if len(num_missing_cols) > 0:
    median_imputer = SimpleImputer(strategy='median')
    glake[num_missing_cols] = median_imputer.fit_transform(glake[num_missing_cols])

print(f'Imputed numeric columns (median): {len(num_missing_cols)}')

# 2) Build ordered area columns for temporal expansion/shrinking features
# Prefer *_km2 when both raw and km2 versions exist for the same year.
area_candidates = [
    c for c in glake.columns
    if re.match(r'^area_\d{4}(_km2)?$', c)
]

year_to_area_col = {}
for c in area_candidates:
    year = int(re.search(r'(\d{4})', c).group(1))
    if year not in year_to_area_col or c.endswith('_km2'):
        year_to_area_col[year] = c

years = sorted(year_to_area_col.keys())
area_cols_used = [year_to_area_col[y] for y in years]

if len(area_cols_used) < 2:
    raise ValueError('Not enough temporal area columns found to compute expansion/shrinking features.')

print('Area columns used:', area_cols_used)

# 3) Interval-wise expansion/shrinking features
for i in range(1, len(area_cols_used)):
    c_prev, c_curr = area_cols_used[i - 1], area_cols_used[i]
    y_prev, y_curr = years[i - 1], years[i]
    dt = y_curr - y_prev

    delta = glake[c_curr] - glake[c_prev]

    glake[f'area_exp_{y_prev}_{y_curr}'] = delta
    glake[f'area_shrink_{y_prev}_{y_curr}'] = (-delta).clip(lower=0)

    denom = glake[c_prev].replace(0, np.nan)
    glake[f'area_exp_pct_{y_prev}_{y_curr}'] = (delta / denom) * 100.0

    valid = (glake[c_prev] > 0) & (glake[c_curr] >= 0)
    glake[f'area_cagr_pct_{y_prev}_{y_curr}'] = np.nan
    glake.loc[valid, f'area_cagr_pct_{y_prev}_{y_curr}'] = (
        (glake.loc[valid, c_curr] / glake.loc[valid, c_prev]) ** (1.0 / dt) - 1.0
    ) * 100.0

# 4) Overall temporal features
first_col, last_col = area_cols_used[0], area_cols_used[-1]
first_year, last_year = years[0], years[-1]

glake['area_exp_total'] = glake[last_col] - glake[first_col]
glake['area_shrink_total'] = (glake[first_col] - glake[last_col]).clip(lower=0)

base = glake[first_col].replace(0, np.nan)
glake['area_exp_total_pct'] = ((glake[last_col] - glake[first_col]) / base) * 100.0

dt_total = last_year - first_year
valid_total = (glake[first_col] > 0) & (glake[last_col] >= 0)
glake['area_cagr_total_pct'] = np.nan
glake.loc[valid_total, 'area_cagr_total_pct'] = (
    (glake.loc[valid_total, last_col] / glake.loc[valid_total, first_col]) ** (1.0 / dt_total) - 1.0
) * 100.0

# 5) Count expansion vs shrinking intervals
delta_matrix = glake[area_cols_used].diff(axis=1).iloc[:, 1:]
glake['n_expanding_intervals'] = (delta_matrix > 0).sum(axis=1)
glake['n_shrinking_intervals'] = (delta_matrix < 0).sum(axis=1)
glake['max_interval_expansion'] = delta_matrix.max(axis=1)
glake['max_interval_shrink'] = delta_matrix.min(axis=1)

# 6) Clean inf values from ratio-based features
ratio_cols = [c for c in glake.columns if ('_pct' in c or 'cagr' in c)]
glake[ratio_cols] = glake[ratio_cols].replace([np.inf, -np.inf], np.nan)

print('\nSample engineered columns:')
print(glake[[
    'area_exp_total', 'area_shrink_total', 'area_exp_total_pct', 'area_cagr_total_pct',
    'n_expanding_intervals', 'n_shrinking_intervals', 'max_interval_expansion', 'max_interval_shrink'
]].head())

Imputed numeric columns (median): 32
Area columns used: ['area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2', 'area_2020_km2']

Sample engineered columns:
   area_exp_total  area_shrink_total  area_exp_total_pct  area_cagr_total_pct  \
0       -0.037518           0.037518          -35.378057            -1.444848   
1        0.019863           0.000000           51.516044             1.394708   
2        0.094178           0.000000           72.635145             1.836698   
3        0.037021           0.000000           99.769839             2.333461   
4        0.131554           0.000000          342.454202             5.082149   

   n_expanding_intervals  n_shrinking_intervals  max_interval_expansion  \
0                      1                      3                0.007704   
1                      2                      2                0.033860   
2                      4                      0                0.036911   
3                      3                   

In [44]:
dam = gpd.read_file(r'dam_metrics_final_fix.csv')

In [45]:
# Add watershed_area_km2 from lake_clean to glake by matching lake ID

# Resolve ID column names robustly
glake_id_col = 'GLAKE_ID' if 'GLAKE_ID' in glake.columns else 'glake_id'
lake_clean_id_col = 'GLAKE_ID' if 'GLAKE_ID' in lake_clean.columns else 'glake_id'

# Build lookup (drop duplicate IDs, keep first)
ws_lookup = (
    lake_clean[[lake_clean_id_col, 'watershed_area_km2']]
    .dropna(subset=[lake_clean_id_col])
    .drop_duplicates(subset=[lake_clean_id_col], keep='first')
    .set_index(lake_clean_id_col)['watershed_area_km2']
)

# Create/overwrite target column in glake
glake['watershed_area_km2'] = glake[glake_id_col].map(ws_lookup)

print("watershed_area_km2 added to glake")
print("Matched rows:", glake['watershed_area_km2'].notna().sum(), "/", len(glake))

watershed_area_km2 added to glake
Matched rows: 6922 / 6923


In [48]:
glake = glake.dropna(subset=['watershed_area_km2']).copy()
print(glake['watershed_area_km2'].isna().sum())

0


In [50]:
glake.to_file('lakes_imputed.gpkg', driver='GPKG')
print('file exported')

file exported


In [12]:
glake.columns

Index(['Type', 'Elevation', 'GLAKE_ID', 'area_2020', 'perimeter_2020',
       'area_1990', 'perimeter_1990', 'geometry_1990', 'area_2000',
       'perimeter_2000', 'geometry_2000', 'area_2010', 'perimeter_2010',
       'geometry_2010', 'area_2015', 'perimeter_2015', 'geometry_2015',
       'area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2',
       'area_2020_km2', 'expansion_rate_km2_yr', 'expansion_rate_pct_yr',
       'glof_happened', 'glof_count', 'lake_type', 'distance_to_glacier_m',
       'nearest_rgiid', 'is_connected', 'wse_m_dam', 'eq_susceptibility',
       'ls_susceptibility', 'volume_m3', 'max_depth_m', 'surface_elevation_m',
       'area_m2', 'perimeter_m', 'RGIId', 'G11_mean_slope_deg',
       'area_exp_1990_2000', 'area_exp_pct_1990_2000',
       'area_cagr_pct_1990_2000', 'area_exp_2000_2010',
       'area_exp_pct_2000_2010', 'area_cagr_pct_2000_2010',
       'area_exp_2010_2015', 'area_exp_pct_2010_2015',
       'area_cagr_pct_2010_2015', 'area_exp_2015

In [11]:
cols = [ 'dam_crest_m_dam', 'dam_height_m_dam',
       'dam_slope_deg_dam', 'dam_width_m_dam', 'freeboard_m_dam']

glake.drop(columns=cols, inplace=True)
print('Dam-related columns dropped.')

Dam-related columns dropped.


In [ ]:
glake.iloc[45]

In [8]:
dam = gpd.read_file(r'dam_metrics_final_fix.csv')

In [84]:
dam = dam_simple.copy()

In [85]:
dam['.geo']

0       {"type":"Polygon","coordinates":[[[77.17606057...
1       {"type":"Polygon","coordinates":[[[73.87725685...
2       {"type":"Polygon","coordinates":[[[91.80517578...
3       {"type":"Polygon","coordinates":[[[74.87687607...
4       {"type":"Polygon","coordinates":[[[86.06279253...
                              ...                        
6918    {"type":"Polygon","coordinates":[[[99.61444672...
6919    {"type":"Polygon","coordinates":[[[97.79569430...
6920    {"type":"Polygon","coordinates":[[[88.71124008...
6921    {"type":"Polygon","coordinates":[[[95.86312140...
6922    {"type":"Polygon","coordinates":[[[95.53891593...
Name: .geo, Length: 6923, dtype: object

In [88]:
import json
from shapely.geometry import shape

# Convert .geo (GeoJSON text) to shapely geometry
def _to_geom(x):
    if pd.isna(x):
        return None
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None
        try:
            x = json.loads(x)
        except json.JSONDecodeError:
            return None
    if isinstance(x, dict):
        if "geometry" in x and isinstance(x["geometry"], dict):  # Feature-like
            x = x["geometry"]
        try:
            return shape(x)
        except Exception:
            return None
    return None

dam["geometry"] = dam[".geo"].apply(_to_geom)

# GeoJSON coordinates are lon/lat -> assign EPSG:4326 first, then project to glake CRS
dam = gpd.GeoDataFrame(dam, geometry="geometry", crs="EPSG:4326")

if glake.crs is None:
    raise ValueError('glake CRS is missing. Set glake CRS before matching geometries.')

dam = dam.to_crs(glake.crs)

# Safety: if coordinates still look like degrees while CRS is projected, fix label then reproject
b = dam.total_bounds
looks_like_degrees = (
    -180 <= b[0] <= 180 and -180 <= b[2] <= 180 and
    -90 <= b[1] <= 90 and -90 <= b[3] <= 90
)
if looks_like_degrees and getattr(glake.crs, 'is_projected', False):
    dam = dam.set_crs('EPSG:4326', allow_override=True).to_crs(glake.crs)
    print('Dam geometry appeared degree-like after projection; CRS label corrected and reprojected.')

print("Converted .geo to geometry and reprojected dam to glake CRS.")
print("glake CRS:", glake.crs)
print("dam CRS:", dam.crs)
print("Valid dam geometries:", dam.geometry.notna().sum(), "/", len(dam))

Converted .geo to geometry and reprojected dam to glake CRS.
glake CRS: PROJCS["Asia_North_Albers_Equal_Area_Conic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",30],PARAMETER["longitude_of_center",95],PARAMETER["standard_parallel_1",15],PARAMETER["standard_parallel_2",65],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102025"]]
dam CRS: PROJCS["Asia_North_Albers_Equal_Area_Conic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG"

In [92]:
dam.columns

Index(['system:index', 'AREA_1', 'Area', 'BASIN', 'DATE_1', 'Date', 'ELEV',
       'E_AREA', 'Elevation', 'Error', 'GLAKE_ID', 'GLID', 'LAT', 'LON',
       'MG_ID1', 'MG_ID2', 'NewPro', 'Not_RGI', 'PERI', 'Perimeter',
       'REGION_1', 'Region', 'SOURCE_1', 'Source', 'TYPE_1', 'Type', 'UTM',
       'VOLUME', 'area_1990', 'area_1990_', 'area_2000', 'area_2000_',
       'area_2010', 'area_2010_', 'area_2015', 'area_2015_', 'area_2020',
       'area_2020_', 'area_new', 'dam_crest_m', 'dam_metrics_reliable',
       'dam_slope_deg', 'expansio_1', 'expansion_', 'fid', 'first_obse',
       'flag_1990', 'flag_2000', 'flag_2010', 'flag_2015', 'freeboard_m',
       'geometry_1', 'geometry_2', 'geometry_3', 'geometry_4', 'glof_count',
       'glof_happe', 'id_source', 'lake_sourc', 'n_obs_poin', 'perimete_1',
       'perimete_2', 'perimete_3', 'perimete_4', 'perimeter_', 'wse_m', '.geo',
       'geometry'],
      dtype='object')

In [89]:
dam['geometry']

0       POLYGON ((-1472358.9 814701.581, -1472344.506 ...
1       POLYGON ((-1710988.972 1006017.73, -1710988.97...
2       POLYGON ((-295807.652 -252328.311, -295803.181...
3       POLYGON ((-1640544.018 943942.327, -1640542.03...
4       POLYGON ((-823813.712 -190093.557, -823813.215...
                              ...                        
6918    POLYGON ((406032.388 202625.296, 406036.362 20...
6919    POLYGON ((257127.36 -192468.27, 257129.84 -192...
6920    POLYGON ((-580987.205 -223029.687, -580981.743...
6921    POLYGON ((77681.749 -944.624, 77688.201 -960.5...
6922    POLYGON ((48350.365 26736.28, 48357.314 26709....
Name: geometry, Length: 6923, dtype: geometry

In [94]:
dam['dam_slope_deg'].describe()

count    6923.000000
mean        0.725367
std         0.407189
min         0.020436
25%         0.333654
50%         0.711421
75%         1.078896
max         1.679715
Name: dam_slope_deg, dtype: float64

In [93]:
cols = ['dam_crest_m', 'dam_metrics_reliable',
       'dam_slope_deg', 'freeboard_m'
       ]
for i in range(len(cols)):
    dam[cols[i]] = pd.to_numeric(dam[cols[i]], errors='coerce')
print('Dam-related columns converted to numeric with coercion.')

Dam-related columns converted to numeric with coercion.


In [82]:
glake.dtypes

Type                     object
Elevation               float64
GLAKE_ID                 object
area_2020               float64
perimeter_2020          float64
                         ...   
dam_metrics_reliable      int64
dam_slope_deg           float64
dam_width_m             float64
freeboard_m             float64
n_obs_poin              float64
Length: 79, dtype: object

In [87]:
glake['dam_slope_deg'].describe()

count    6922.000000
mean        2.932811
std        11.439236
min       -53.370505
25%         0.672254
50%         3.777251
75%         8.699836
max        47.460675
Name: dam_slope_deg, dtype: float64

In [53]:
import geopandas as gpd
print("glake CRS:", glake.crs)
print("dam CRS:", dam.crs)
print("glake bounds:", glake.total_bounds)
print("dam bounds:", dam.total_bounds)
print("valid glake:", glake.geometry.notna().sum(), "/", len(glake))
print("valid dam:", dam.geometry.notna().sum(), "/", len(dam))

glake CRS: PROJCS["Asia_North_Albers_Equal_Area_Conic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",30],PARAMETER["longitude_of_center",95],PARAMETER["standard_parallel_1",15],PARAMETER["standard_parallel_2",65],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102025"]]
dam CRS: PROJCS["Asia_North_Albers_Equal_Area_Conic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_

In [62]:
# Fresh geometry-only transfer from dam -> glake
import geopandas as gpd
import pandas as pd

dam_add_cols = [
    'dam_crest_m', 'dam_height_m', 'dam_metrics_reliable',
    'dam_slope_deg', 'dam_width_m', 'freeboard_m', 'n_obs_poin'
]

# Checks
missing_in_dam = [c for c in dam_add_cols if c not in dam.columns]
if missing_in_dam:
    raise KeyError(f"Missing columns in dam: {missing_in_dam}")
if 'geometry' not in glake.columns or 'geometry' not in dam.columns:
    raise KeyError("Both glake and dam must have geometry columns.")

# Ensure same CRS for geometric matching
if glake.crs is not None and dam.crs is not None and glake.crs != dam.crs:
    dam = dam.to_crs(glake.crs)

# Use only geometry + requested dam attributes
dam_src = dam[['geometry'] + dam_add_cols].copy()
dam_src = dam_src[dam_src.geometry.notna()].copy()
glake_src = glake[['geometry']].copy()
glake_src = glake_src[glake_src.geometry.notna()].copy()

# Pure geometry-based join (no ID/key columns involved)
joined = gpd.sjoin(
    glake_src,
    dam_src,
    how='left',
    predicate='intersects'
)

matched_rows = joined['index_right'].notna().sum()
matched_lakes = joined.index[joined['index_right'].notna()].nunique()
print(f'glake CRS: {glake.crs}')
print(f'dam CRS: {dam.crs}')
print(f'matched lake rows: {matched_lakes}')
print(f'matched geometry rows: {matched_rows}')

if matched_rows == 0:
    raise ValueError(
        'No intersecting geometries were found between glake and dam. '
        'This means the join produced no matches, so zero-filling would make every value 0. '
        'Check that both layers use the same CRS, that geometries are valid, and that intersects is the right predicate.'
    )

# Aggregate if one lake intersects multiple dam geometries
agg_map = {}
for c in dam_add_cols:
    if pd.api.types.is_numeric_dtype(dam_src[c]) or pd.api.types.is_bool_dtype(dam_src[c]):
        agg_map[c] = 'mean'
    else:
        agg_map[c] = 'first'

geom_summary = joined.groupby(joined.index).agg(agg_map)

# Replace/add these columns in glake from geometry summary
glake = glake.drop(columns=dam_add_cols, errors='ignore')
glake = glake.join(geom_summary, how='left')

# Unmatched geometries -> 0
for c in dam_add_cols:
    glake[c] = glake[c].fillna(0)

print('Geometry-only dam feature transfer completed.')
print('NaN counts in dam_add_cols after fill:')
print(glake[dam_add_cols].isna().sum())

glake CRS: PROJCS["Asia_North_Albers_Equal_Area_Conic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",30],PARAMETER["longitude_of_center",95],PARAMETER["standard_parallel_1",15],PARAMETER["standard_parallel_2",65],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102025"]]
dam CRS: PROJCS["Asia_North_Albers_Equal_Area_Conic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_

In [63]:
glake['dam_crest_m']

0       4551.293148118622
1       5276.284357244318
2       5315.128231272976
3       5390.186369567357
4       5278.600353033474
              ...        
6917    4626.817432315667
6918     4759.91356102196
6919    4448.257047954358
6920    5498.586147072041
6921    5409.986131959608
Name: dam_crest_m, Length: 6922, dtype: object

In [32]:
glake.columns

Index(['Type', 'Elevation', 'GLAKE_ID', 'area_2020', 'perimeter_2020',
       'area_1990', 'perimeter_1990', 'geometry_1990', 'area_2000',
       'perimeter_2000', 'geometry_2000', 'area_2010', 'perimeter_2010',
       'geometry_2010', 'area_2015', 'perimeter_2015', 'geometry_2015',
       'area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2',
       'area_2020_km2', 'expansion_rate_km2_yr', 'expansion_rate_pct_yr',
       'glof_happened', 'glof_count', 'lake_type', 'distance_to_glacier_m',
       'nearest_rgiid', 'is_connected', 'wse_m_dam', 'eq_susceptibility',
       'ls_susceptibility', 'volume_m3', 'max_depth_m', 'surface_elevation_m',
       'area_m2', 'perimeter_m', 'RGIId', 'G11_mean_slope_deg',
       'area_exp_1990_2000', 'area_exp_pct_1990_2000',
       'area_cagr_pct_1990_2000', 'area_exp_2000_2010',
       'area_exp_pct_2000_2010', 'area_cagr_pct_2000_2010',
       'area_exp_2010_2015', 'area_exp_pct_2010_2015',
       'area_cagr_pct_2010_2015', 'area_exp_2015

In [ ]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Additional shape + temporal stability features
# ------------------------------------------------------------

# 1) Shape/compactness and area-perimeter ratio from geometry
# Project to a metric CRS so area/perimeter are in m²/m.
if glake.crs is None:
    raise ValueError('glake has no CRS; cannot compute metric geometry features reliably.')

if getattr(glake.crs, 'is_geographic', False):
    metric_crs = glake.estimate_utm_crs()
    if metric_crs is None:
        raise ValueError('Could not estimate a projected CRS for metric geometry calculations.')
    _g_metric = glake.to_crs(metric_crs)
else:
    _g_metric = glake

area_m2 = _g_metric.geometry.area
perimeter_m = _g_metric.geometry.length

# compactness = 4*pi*area / perimeter^2
glake['compactness'] = np.where(
    perimeter_m > 0,
    (4.0 * np.pi * area_m2) / (perimeter_m ** 2),
    np.nan
)

# area_perimeter_ratio = area_m2 / perimeter_m
glake['area_perimeter_ratio'] = np.where(
    perimeter_m > 0,
    area_m2 / perimeter_m,
    np.nan
)

# 2) Elevation normalized (quantile rank in [0, 1])
elev_candidates = [
    c for c in glake.columns
    if ('elev' in c.lower()) and pd.api.types.is_numeric_dtype(glake[c])
]

if len(elev_candidates) == 0:
    glake['elevation_normalized'] = np.nan
    print('No numeric elevation column found; elevation_normalized set to NaN.')
else:
    elev_col = elev_candidates[0]
    glake['elevation_normalized'] = glake[elev_col].rank(pct=True, method='average')
    print(f'Using elevation column: {elev_col}')

# 3) Expansion consistency
if {'n_expanding_intervals', 'n_shrinking_intervals'}.issubset(glake.columns):
    total_moves = glake['n_expanding_intervals'] + glake['n_shrinking_intervals']
    glake['expansion_consistency'] = np.where(
        total_moves > 0,
        glake['n_expanding_intervals'] / total_moves,
        0.0
    )
else:
    glake['expansion_consistency'] = np.nan
    print('n_expanding_intervals / n_shrinking_intervals not found; expansion_consistency set to NaN.')

# 4) Absolute area volatility across target years
target_years = [1990, 2000, 2010, 2015, 2020]
area_cols = []
for y in target_years:
    km2_col = f'area_{y}_km2'
    raw_col = f'area_{y}'
    if km2_col in glake.columns:
        area_cols.append(km2_col)
    elif raw_col in glake.columns:
        area_cols.append(raw_col)

if len(area_cols) < 2:
    glake['area_std_across_years'] = np.nan
    glake['area_cv_across_years'] = np.nan
    print('Insufficient area year columns found; area_std_across_years and area_cv_across_years set to NaN.')
else:
    area_block = glake[area_cols].apply(pd.to_numeric, errors='coerce')
    glake['area_std_across_years'] = area_block.std(axis=1, ddof=0)
    area_mean = area_block.mean(axis=1)
    glake['area_cv_across_years'] = np.where(
        area_mean > 0,
        glake['area_std_across_years'] / area_mean,
        np.nan
    )
    print('Area volatility columns computed from:', area_cols)

print('New features added:')
print([
    'compactness',
    'area_perimeter_ratio',
    'elevation_normalized',
    'expansion_consistency',
    'area_std_across_years',
    'area_cv_across_years',
])
print(glake[[
    'compactness',
    'area_perimeter_ratio',
    'elevation_normalized',
    'expansion_consistency',
    'area_std_across_years',
    'area_cv_across_years',
]].head())

Using elevation column: Elevation
Area volatility columns computed from: ['area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2', 'area_2020_km2']
New features added:
['compactness', 'area_perimeter_ratio', 'elevation_normalized', 'expansion_consistency', 'area_std_across_years', 'area_cv_across_years']
   compactness  area_perimeter_ratio  elevation_normalized  \
0     0.563186             55.419975              0.238948   
1     0.431855             44.806893              0.738298   
2     0.320314             75.535129              0.782722   
3     0.850582             70.834230              0.838630   
4     0.662047             94.629039              0.749711   

   expansion_consistency  area_std_across_years  area_cv_across_years  
0                   0.25               0.019921              0.211174  
1                   0.50               0.011625              0.210115  
2                   1.00               0.036962              0.211626  
3                   0.

In [97]:
glake.to_file('model_data\Final data\lakes_imputed_22.gpkg', driver='GPKG')

<>:1: SyntaxWarning: invalid escape sequence '\F'
<>:1: SyntaxWarning: invalid escape sequence '\F'
C:\Users\rubel\AppData\Local\Temp\ipykernel_31828\3436296761.py:1: SyntaxWarning: invalid escape sequence '\F'
  glake.to_file('model_data\Final data\lakes_imputed_22.gpkg', driver='GPKG')


In [35]:
dam_simple = gpd.read_file(r'dam_metrics_simple.csv')
dam_simple.columns

Index(['system:index', 'AREA_1', 'Area', 'BASIN', 'DATE_1', 'Date', 'ELEV',
       'E_AREA', 'Elevation', 'Error', 'GLAKE_ID', 'GLID', 'LAT', 'LON',
       'MG_ID1', 'MG_ID2', 'NewPro', 'Not_RGI', 'PERI', 'Perimeter',
       'REGION_1', 'Region', 'SOURCE_1', 'Source', 'TYPE_1', 'Type', 'UTM',
       'VOLUME', 'area_1990', 'area_1990_', 'area_2000', 'area_2000_',
       'area_2010', 'area_2010_', 'area_2015', 'area_2015_', 'area_2020',
       'area_2020_', 'area_new', 'dam_crest_m', 'dam_metrics_reliable',
       'dam_slope_deg', 'expansio_1', 'expansion_', 'fid', 'first_obse',
       'flag_1990', 'flag_2000', 'flag_2010', 'flag_2015', 'freeboard_m',
       'geometry_1', 'geometry_2', 'geometry_3', 'geometry_4', 'glof_count',
       'glof_happe', 'id_source', 'lake_sourc', 'n_obs_poin', 'perimete_1',
       'perimete_2', 'perimete_3', 'perimete_4', 'perimeter_', 'wse_m',
       '.geo'],
      dtype='object')

In [66]:
glake['dam_slope_deg'].info

<bound method Series.info of 0        3.176274993725146
1       13.484769682231828
2       2.3072190276745967
3        2.595260542130418
4        3.677171348803358
               ...        
6917    0.9761475456632672
6918    16.635806113819566
6919    2.4620137925336185
6920     4.104731721729393
6921    12.781098809482415
Name: dam_slope_deg, Length: 6922, dtype: object>

In [57]:
# Compact CRS/extent diagnostic
print('glake crs:', glake.crs)
print('dam crs:', dam.crs)

print('glake bounds:', [round(float(v), 3) for v in glake.total_bounds])
print('dam bounds:', [round(float(v), 3) for v in dam.total_bounds])

g4326 = glake.to_crs(4326) if glake.crs is not None else glake
d4326 = dam.to_crs(4326) if dam.crs is not None else dam
print('glake bounds (4326):', [round(float(v), 6) for v in g4326.total_bounds])
print('dam bounds (4326):', [round(float(v), 6) for v in d4326.total_bounds])

gb = g4326.total_bounds
db = d4326.total_bounds
overlap = not (gb[2] < db[0] or gb[0] > db[2] or gb[3] < db[1] or gb[1] > db[3])
print('bbox overlap in 4326:', overlap)

glake crs: PROJCS["Asia_North_Albers_Equal_Area_Conic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",30],PARAMETER["longitude_of_center",95],PARAMETER["standard_parallel_1",15],PARAMETER["standard_parallel_2",65],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102025"]]
dam crs: PROJCS["Asia_North_Albers_Equal_Area_Conic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_

In [96]:
# Fix only negative dam_slope_deg in glake using geometry-based rematch from dam
import pandas as pd
import geopandas as gpd

if 'dam_slope_deg' not in glake.columns:
    raise KeyError("glake is missing 'dam_slope_deg'")
if 'dam_slope_deg' not in dam.columns:
    raise KeyError("dam is missing 'dam_slope_deg'")
if 'geometry' not in glake.columns or 'geometry' not in dam.columns:
    raise KeyError("Both glake and dam must have geometry columns")

# Ensure numeric slope columns for safe filtering/assignment
glake['dam_slope_deg'] = pd.to_numeric(glake['dam_slope_deg'], errors='coerce')
dam_slope_num = pd.to_numeric(dam['dam_slope_deg'], errors='coerce')

# Work on a CRS-aligned copy of dam
dam_fix = dam.copy()
if glake.crs is not None and dam_fix.crs is not None and glake.crs != dam_fix.crs:
    dam_fix = dam_fix.to_crs(glake.crs)

# Only rows needing correction
neg_mask = glake['dam_slope_deg'] < 0
n_neg = int(neg_mask.sum())
print(f'Rows with negative dam_slope_deg before fix: {n_neg}')

if n_neg > 0:
    left = glake.loc[neg_mask, ['geometry']].copy()
    right = gpd.GeoDataFrame(
        {'dam_slope_deg_src': dam_slope_num, 'geometry': dam_fix.geometry},
        geometry='geometry',
        crs=dam_fix.crs
    )
    right = right[right.geometry.notna()].copy()

    # Geometry-only match
    sjoin_fix = gpd.sjoin(left, right, how='left', predicate='intersects')

    # Aggregate source slope per glake row (multiple matches -> mean)
    slope_map = sjoin_fix.groupby(sjoin_fix.index)['dam_slope_deg_src'].mean()

    # Update ONLY dam_slope_deg and ONLY on negative rows with a matched value
    target_idx = glake.index[neg_mask]
    replacement = slope_map.reindex(target_idx)
    can_update = replacement.notna()
    glake.loc[target_idx[can_update], 'dam_slope_deg'] = replacement.loc[can_update].values

n_neg_after = int((glake['dam_slope_deg'] < 0).sum())
print(f'Rows with negative dam_slope_deg after fix: {n_neg_after}')

Rows with negative dam_slope_deg before fix: 0
Rows with negative dam_slope_deg after fix: 0


In [131]:
lakes = gpd.read_file('model_data\Final data\lakes_imputed_24.gpkg')
length = gpd.read_file('model_data\glacier_contact\glacier_contact.shp')
velocity = gpd.read_file('glacier_velocities_composite.csv')

<>:1: SyntaxWarning: invalid escape sequence '\F'
<>:2: SyntaxWarning: invalid escape sequence '\g'
<>:1: SyntaxWarning: invalid escape sequence '\F'
<>:2: SyntaxWarning: invalid escape sequence '\g'
C:\Users\rubel\AppData\Local\Temp\ipykernel_31828\1874503924.py:1: SyntaxWarning: invalid escape sequence '\F'
  lakes = gpd.read_file('model_data\Final data\lakes_imputed_24.gpkg')
C:\Users\rubel\AppData\Local\Temp\ipykernel_31828\1874503924.py:2: SyntaxWarning: invalid escape sequence '\g'
  length = gpd.read_file('model_data\glacier_contact\glacier_contact.shp')


In [101]:
velocity.columns

Index(['RGIId', 'v_mean_2020'], dtype='object')

In [133]:
lakes.columns

Index(['Type', 'Elevation', 'GLAKE_ID', 'area_2020', 'perimeter_2020',
       'area_1990', 'perimeter_1990', 'geometry_1990', 'area_2000',
       'perimeter_2000', 'geometry_2000', 'area_2010', 'perimeter_2010',
       'geometry_2010', 'area_2015', 'perimeter_2015', 'geometry_2015',
       'area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2',
       'area_2020_km2', 'expansion_rate_km2_yr', 'expansion_rate_pct_yr',
       'glof_happened', 'glof_count', 'lake_type', 'distance_to_glacier_m',
       'nearest_rgiid', 'is_connected', 'wse_m_dam', 'eq_susceptibility',
       'ls_susceptibility', 'volume_m3', 'max_depth_m', 'surface_elevation_m',
       'area_m2', 'perimeter_m', 'RGIId', 'G11_mean_slope_deg',
       'area_exp_1990_2000', 'area_exp_pct_1990_2000',
       'area_cagr_pct_1990_2000', 'area_exp_2000_2010',
       'area_exp_pct_2000_2010', 'area_cagr_pct_2000_2010',
       'area_exp_2010_2015', 'area_exp_pct_2010_2015',
       'area_cagr_pct_2010_2015', 'area_exp_2015

In [134]:
lakes.drop(columns = ['contact_m', 'v_mean_2020'], inplace=True)

In [135]:
# Add contact_m from length -> lakes (match on glake_id/GLAKE_ID), fill unmatched with 0 
lakes_id_col = 'GLAKE_ID' if 'GLAKE_ID' in lakes.columns else 'glake_id'
length_id_col = 'GLAKE_ID' if 'GLAKE_ID' in length.columns else 'glake_id'

contact_lookup = (
    length[[length_id_col, 'contact_m']]
    .dropna(subset=[length_id_col])
    .drop_duplicates(subset=[length_id_col], keep='first')
    .set_index(length_id_col)['contact_m']
)
lakes['contact_m'] = lakes[lakes_id_col].map(contact_lookup).fillna(0) 
 
# Add v_mean_2020 from velocity -> lakes (match on RGIId) and impute missing with median 
velocity['v_mean_2020'] = pd.to_numeric(velocity['v_mean_2020'], errors='coerce') 
v_mean_2020_median = velocity['v_mean_2020'].median() 
velocity_lookup = ( 
    velocity[['RGIId', 'v_mean_2020']] 
    .dropna(subset=['RGIId']) 
    .drop_duplicates(subset=['RGIId'], keep='first') 
    .set_index('RGIId')['v_mean_2020'] 
) 
lakes['v_mean_2020'] = lakes['RGIId'].map(velocity_lookup).fillna(v_mean_2020_median) 
 
print('Done. Added/updated columns: contact_m, v_mean_2020') 
print('Unmatched lakes for contact_m:', int((lakes['contact_m'] == 0).sum())) 
print('Matched lakes for v_mean_2020:', int(lakes['v_mean_2020'].notna().sum())) 
print('Imputed v_mean_2020 nulls with median:', float(v_mean_2020_median)) 
print('Remaining null v_mean_2020:', int(lakes['v_mean_2020'].isna().sum())) 

Done. Added/updated columns: contact_m, v_mean_2020
Unmatched lakes for contact_m: 5683
Matched lakes for v_mean_2020: 6922
Imputed v_mean_2020 nulls with median: 0.7490594046456474
Remaining null v_mean_2020: 0


In [124]:
lakes['v_mean_2020'].isna().sum()

np.int64(4)

In [136]:
import pandas as pd

lakes['v_mean_2020'] = pd.to_numeric(lakes['v_mean_2020'], errors='coerce').astype('float32')

print('v_mean_2020 stats:', lakes['v_mean_2020'].describe())

v_mean_2020 stats: count    6922.000000
mean        2.515817
std         7.230976
min         0.074328
25%         0.450548
50%         0.736582
75%         1.496928
max       169.421326
Name: v_mean_2020, dtype: float64


In [137]:
lakes['v_mean_2020'].describe()

count    6922.000000
mean        2.515817
std         7.230976
min         0.074328
25%         0.450548
50%         0.736582
75%         1.496928
max       169.421326
Name: v_mean_2020, dtype: float64

In [138]:
lakes.to_file('model_data\Final data\lakes_imputed_24.gpkg', driver='GPKG')

<>:1: SyntaxWarning: invalid escape sequence '\F'
<>:1: SyntaxWarning: invalid escape sequence '\F'
C:\Users\rubel\AppData\Local\Temp\ipykernel_31828\2097505895.py:1: SyntaxWarning: invalid escape sequence '\F'
  lakes.to_file('model_data\Final data\lakes_imputed_24.gpkg', driver='GPKG')
